RAG system starts by using vector representations of you text to match you prompt to relevant sections within the unstructured data.

In order to be able to find relevant text in a knowledge graph in the same way, you will need to create embeddings of the text fields of your graph.




In [1]:
from dotenv import load_dotenv
import os

from langchain_community.graphs import Neo4jGraph

In [2]:
load_dotenv()

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE")

In [3]:
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [4]:
kg = Neo4jGraph(url= NEO4J_URI, username=NEO4J_USERNAME, password=NEO4J_PASSWORD, database=NEO4J_DATABASE)

/var/folders/s8/qyjb36g92fs3ztdqk120mmkw0000gn/T/ipykernel_7856/2155551757.py:1: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-neo4j package and should be used instead. To use it run `pip install -U :class:`~langchain-neo4j` and import as `from :class:`~langchain_neo4j import Neo4jGraph``.
  kg = Neo4jGraph(url= NEO4J_URI, username=NEO4J_USERNAME, password=NEO4J_PASSWORD, database=NEO4J_DATABASE)


In [5]:
# openai embedding models default vector dimension size is 1536 for OpenAI text-embedding-ada-002

kg.query("""
    CREATE VECTOR INDEX movie_tagline_embeddings IF NOT EXISTS
    FOR (m:movie) ON (m.taglineEmbedding)     
    OPTIONS { indexConfig: {
        `vector.dimensions`:1536,
        `vector.similarity_function`:'cosine'
    }}     
""")

[]

In [6]:
# to check vector indexes


kg.query(""" 
         SHOW VECTOR INDEXES
         """
)

[{'id': 6,
  'name': 'movie_tagline_embeddings',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'VECTOR',
  'entityType': 'NODE',
  'labelsOrTypes': ['movie'],
  'properties': ['taglineEmbedding'],
  'indexProvider': 'vector-2.0',
  'owningConstraint': None,
  'lastRead': neo4j.time.DateTime(2025, 6, 16, 13, 55, 52, 570000000, tzinfo=<UTC>),
  'readCount': 20}]

In [7]:
kg.query(""" 
         MATCH (movie:Movie) WHERE movie.tagline IS NOT NULL
         WITH movie, genai.vector.encode(
             movie.tagline,
             "OpenAI",
             {token: $openAiApiKey}
            )AS vector
         CALL db.create.setNodeVectorProperty(movie, "taglineEmbedding", vector)
         """,
    params = {"openAiApiKey": os.getenv("OPENAI_API_KEY")}
)

[]

In [8]:
result = kg.query("""
                  MATCH (m:Movie)
                  WHERE m.tagline IS NOT NULL
                  RETURN m.tagline, m.taglineEmbedding
                  LIMIT 1
                  """)

In [9]:
result[0]['m.tagline']

'Welcome to the Real World'

In [10]:
result[0]['m.taglineEmbedding'][:10]

[0.01746697723865509,
 -0.005447783973067999,
 -0.0020621647126972675,
 -0.025564946234226227,
 -0.014322134666144848,
 0.016733180731534958,
 -0.017047664150595665,
 0.0004647650639526546,
 -0.025211151689291,
 -0.029509101063013077]

In [11]:
len(result[0]['m.taglineEmbedding'])

1536

In [12]:
# As we have embedding for every movie in database, we can actually  use cosine similarity search on that movies

#similarity search as we done vector embeddings on taglines and hence question will be on taglines

question = "What movies are about love?"

In [13]:
# top_k : we just want top k results instead of returning all results we want similarity for top_k specified in parameters.

result = kg.query("""
                    WITH genai.vector.encode(
                        $question,
                        "OpenAI",
                        {token: $openAiApiKey}) AS question_embedding
                    CALL db.index.vector.queryNodes(
                        'movie_tagline_embeddings',
                        $top_k,
                        question_embedding
                    ) YIELD node AS movie, score
                    RETURN movie.title, movie.tagline, score     
                    """,
                    params = {"openAiApiKey": os.getenv("OPENAI_API_KEY"),
                            "question":question,
                            "top_k": 5}
                    )

In [14]:
print(result)

[]
